In [ ]:
import sys
from pathlib import Path

# ---------------------------------------------------------
# 1. BOOTSTRAP: Add ../src to the path to import config
# This assumes main.py is in a directory like $root/notebooks/
# ---------------------------------------------------------
src_path = str(Path.cwd().parent / "src")
if src_path not in sys.path:
    sys.path.insert(0, src_path)

import config  # This automatically sets the working directory to the project root

# ---------------------------------------------------------
# 2. STANDARD IMPORTS
# ---------------------------------------------------------
import pandas as pd
import json
import os
from pathlib import Path

# --- Corrected Paths ---
# Since you are in /volleyball_st_james, we look directly into 'Data'
BASE_DIR = Path('Data')
RAW_EVENTS_DIR = BASE_DIR / 'raw/events'
RAW_RESULTS_DIR = BASE_DIR / 'raw/event_results'
PROCESSED_DIR = BASE_DIR / 'processed'

# Ensure output directories exist
for folder in ['events', 'clubs', 'divisions']:
    (PROCESSED_DIR / folder).mkdir(parents=True, exist_ok=True)

all_events = []
all_matches = []
all_divisions = []
all_clubs = []
mismatches = []

print(f"Mission Start: Scanning {RAW_EVENTS_DIR}...")

# --- Process JSONs ---
json_files = list(RAW_EVENTS_DIR.glob('*.json'))
print(f"Found {len(json_files)} JSON files.")

for json_file in json_files:
    with open(json_file, 'r') as f:
        event_data = json.load(f)
    
    e_id = event_data.get('EventId')
    e_key = event_data.get('Key')
    
    # Event metadata
    all_events.append({
        'EventId': e_id,
        'Name': event_data.get('Name'),
        'StartDate': event_data.get('StartDate'),
        'Location': event_data.get('Location'),
        'IsOver': event_data.get('IsOver')
    })

    # Divisions & Clubs
    for div in event_data.get('Divisions', []):
        div['EventId'] = e_id
        all_divisions.append(div)
    for club in event_data.get('Clubs', []):
        club['EventId'] = e_id
        all_clubs.append(club)

    # Link CSV
    csv_path = RAW_RESULTS_DIR / f"{e_key}.csv"
    if csv_path.exists():
        try:
            # Using sep=None allows pandas to guess if it's a comma or tab
            df_m = pd.read_csv(csv_path, sep=None, engine='python')
            df_m['EventId'] = e_id
            all_matches.append(df_m)
        except Exception as e:
            print(f"Warning: Could not read CSV for Event {e_id}: {e}")
    else:
        mismatches.append({'EventId': e_id, 'Name': event_data.get('Name'), 'Key': e_key})

# --- Consolidation & Export ---
df_events = pd.DataFrame(all_events)
df_all_divisions = pd.DataFrame(all_divisions)
df_all_clubs = pd.DataFrame(all_clubs)

df_events.to_csv(PROCESSED_DIR / 'events/master_events.csv', index=False)
df_all_divisions.to_csv(PROCESSED_DIR / 'divisions/master_divisions.csv', index=False)
df_all_clubs.to_csv(PROCESSED_DIR / 'clubs/master_clubs.csv', index=False)

if all_matches:
    df_all_matches = pd.concat(all_matches, ignore_index=True)
    df_all_matches.to_csv(PROCESSED_DIR / 'events/master_match_results.csv', index=False)
    print(f"Successfully processed {len(all_matches)} result files.")

if mismatches:
    pd.DataFrame(mismatches).to_csv(PROCESSED_DIR / 'mismatch_report.csv', index=False)
    print(f"Mission Note: {len(mismatches)} events missing results. Report saved to processed/mismatch_report.csv")

print("--- Data Consolidation Complete ---")


In [ ]:
import os
from pathlib import Path

# Check paths
raw_events = Path('../Data/raw/events')
raw_results = Path('../Data/raw/event_results')

json_files = list(raw_events.glob('*.json'))
csv_files = list(raw_results.glob('*.csv'))

print("--- File Count Diagnostic ---")
print(f"JSON files found: {len(json_files)}")
print(f"CSV files found:  {len(csv_files)}")

if len(json_files) > 0:
    print("\n--- Sample Check ---")
    sample_json = json_files[0]
    print(f"Sample JSON filename: {sample_json.name}")
    
    # Check what key is inside
    import json
    with open(sample_json, 'r') as f:
        data = json.load(f)
        key = data.get('Key')
        print(f"Key found inside JSON: {key}")
        
        expected_csv = raw_results / f"{key}.csv"
        print(f"Looking for CSV at: {expected_csv}")
        print(f"Does that CSV exist? {expected_csv.exists()}")


In [ ]:
import os
from pathlib import Path

# 1. Print where the notebook is currently 'standing'
print(f"Current Working Directory: {os.getcwd()}")

# 2. List folders in the immediate area
print("\nItems in Current Directory:")
print(os.listdir('.'))

# 3. Check one level up (where you expected the Data folder to be)
try:
    print("\nItems in Parent Directory (../):")
    print(os.listdir('..'))
except Exception as e:
    print(f"\nCould not access Parent Directory: {e}")

# 4. Search for the 'Data' folder specifically
print("\nSearching for 'Data' folder...")
found_data = False
for path in Path('.').rglob('Data'):
    print(f"FOUND: {path.absolute()}")
    found_data = True

if not found_data:
    print("CRITICAL: No folder named 'Data' found in this directory or subdirectories.")
